# Complete Breakdown: All Classes and Functions in image_curator.py

This notebook explains **every single class and function** with working examples.

---

## Setup: Import Everything

In [2]:
import json
import os
import argparse
import sys
from pathlib import Path
from PIL import Image
import csv
from typing import Dict, List, Optional

---
---
# CLASS 1: ImageMetadata

## What This Class Does:
The `ImageMetadata` class stores **all information about a single image**. It's like a detailed profile card that holds:
- Basic info (ID, file location)
- Visual characteristics (scene type, complexity, colors)
- Content details (subjects, actions, setting)
- Structured answers (who/what/where/color/size/mood)
- Gold standard description (the perfect answer)

**Think of it as:** A comprehensive information card for one image.

---

## Function 1.1: `__init__()`

### What it does:
Creates a new ImageMetadata object with empty fields ready to be filled.

### Parameters:
- `image_id` (str): Unique identifier for the image
- `file_path` (str): Path to the image file

### What gets initialized:
- Sets the ID and file path
- Creates empty placeholders for all other fields

In [ ]:
# Example 1.1: Create a new ImageMetadata object

metadata = ImageMetadata(
    image_id="beach_001",
    file_path="data/images/beach_sunset.jpg"
)

print("Created ImageMetadata object:")
print(f"  image_id: {metadata.image_id}")
print(f"  file_path: {metadata.file_path}")
print(f"  scene_type: {metadata.scene_type}")  # None
print(f"  complexity_level: {metadata.complexity_level}")  # None
print(f"  primary_subjects: {metadata.primary_subjects}")  # []
print(f"  actions: {metadata.actions}")  # []
print(f"  colors: {metadata.colors}")  # []
print(f"  setting: {metadata.setting}")  # None
print(f"  structure_words: {metadata.structure_words}")  # Dict with empty strings
print(f"  gold_standard: {metadata.gold_standard}")  # None

print("\n✓ Object created with all fields initialized!")

## Function 1.2: `to_dict()`

### What it does:
Converts the ImageMetadata object into a Python dictionary.

### Parameters:
- None (uses the object's own data via `self`)

### Returns:
- Dictionary containing all metadata fields

### Why we need it:
Dictionaries can be saved to JSON files, but objects cannot. This prepares the data for saving.

In [ ]:
# Example 1.2: Convert object to dictionary

# First, create and populate an object
metadata = ImageMetadata("park_001", "data/images/park.jpg")
metadata.scene_type = "outdoor"
metadata.complexity_level = 2
metadata.primary_subjects = ["children", "playground"]
metadata.actions = ["playing", "swinging"]
metadata.colors = ["green", "red", "blue"]
metadata.setting = "city park"
metadata.structure_words["who"] = "children"
metadata.structure_words["what"] = "playing"
metadata.gold_standard = "Children playing in a park."

# Convert to dictionary
metadata_dict = metadata.to_dict()

print("Original object type:", type(metadata))
print("Converted to type:", type(metadata_dict))
print("\nDictionary contents:")
print(json.dumps(metadata_dict, indent=2))

print("\n✓ Object successfully converted to dictionary!")

## Function 1.3: `from_dict()` (Static Method)

### What it does:
Creates a new ImageMetadata object from a dictionary (opposite of `to_dict()`).

### Parameters:
- `data` (Dict): Dictionary containing metadata fields

### Returns:
- New ImageMetadata object with all fields populated

### Why it's static (`@staticmethod`):
You call it on the class itself, not on an existing object. Used when loading from files.

In [ ]:
# Example 1.3: Create object from dictionary

# Simulate data loaded from a JSON file
loaded_data = {
    "image_id": "mountain_001",
    "file_path": "data/images/mountain.jpg",
    "scene_type": "outdoor",
    "complexity_level": 3,
    "primary_subjects": ["mountain", "hiker", "trail"],
    "actions": ["hiking", "climbing"],
    "colors": ["blue", "green", "brown"],
    "setting": "mountain trail",
    "structure_words": {
        "who": "hiker",
        "where": "mountain",
        "what": "hiking"
    },
    "gold_standard": "A hiker on a mountain trail."
}

# Create ImageMetadata object from dictionary
metadata = ImageMetadata.from_dict(loaded_data)

print("Recreated ImageMetadata object:")
print(f"  image_id: {metadata.image_id}")
print(f"  scene_type: {metadata.scene_type}")
print(f"  complexity_level: {metadata.complexity_level}")
print(f"  primary_subjects: {metadata.primary_subjects}")
print(f"  actions: {metadata.actions}")
print(f"  gold_standard: {metadata.gold_standard}")

print("\n✓ Dictionary successfully converted to object!")

## ImageMetadata Summary

| Function | Purpose | Input | Output |
|----------|---------|-------|--------|
| `__init__()` | Create new object | image_id, file_path | ImageMetadata object |
| `to_dict()` | Object → Dictionary | None | Dict |
| `from_dict()` | Dictionary → Object | Dict | ImageMetadata object |

---

---
---
# CLASS 2: Prompt

## What This Class Does:
The `Prompt` class represents **a question about an image**. Each prompt:
- Is tied to a specific image
- Has a structure word (who/what/where/color/size/action/mood)
- Contains the actual question text
- Has a difficulty level (1-3)
- Can use default templates or custom questions

**Think of it as:** A question card for an image with automatic difficulty assignment.

---

## Class Variables (Shared by all Prompt objects)

### DIFFICULTY_MAP
Maps structure words to difficulty levels (1=easy, 2=medium, 3=hard)

In [ ]:
# Example 2.0: View difficulty mappings

print("DIFFICULTY_MAP:")
for word, level in Prompt.DIFFICULTY_MAP.items():
    print(f"  {word:10} → Level {level}")

print("\nDifficulty meanings:")
print("  Level 1: Easy (simple observation)")
print("  Level 2: Medium (understanding required)")
print("  Level 3: Hard (interpretation needed)")

### DEFAULT_TEMPLATES
Pre-written questions for each structure word

In [ ]:
# Example 2.0b: View default templates

print("DEFAULT_TEMPLATES:")
for word, template in Prompt.DEFAULT_TEMPLATES.items():
    print(f"  {word:10} → {template}")

## Function 2.1: `__init__()`

### What it does:
Creates a new Prompt object. Can use default question or provide custom one.

### Parameters:
- `prompt_id` (str): Unique ID for this prompt (e.g., "beach_001_q_who")
- `image_id` (str): ID of the image this prompt is about
- `structure_word` (str): Type of question (who/what/where/etc.)
- `question` (str, optional): Custom question text. If None, uses default template
- `difficulty` (int, optional): Custom difficulty. If None, uses DIFFICULTY_MAP

### Smart features:
- Auto-generates question from template if not provided
- Auto-assigns difficulty based on structure word

In [ ]:
# Example 2.1a: Create prompt with default question

prompt1 = Prompt(
    prompt_id="beach_001_q_who",
    image_id="beach_001",
    structure_word="who"
)

print("Prompt with default question:")
print(f"  prompt_id: {prompt1.prompt_id}")
print(f"  image_id: {prompt1.image_id}")
print(f"  structure_word: {prompt1.structure_word}")
print(f"  question: {prompt1.question}")  # Uses default template
print(f"  difficulty: {prompt1.difficulty}")  # Auto-assigned

print("\n" + "="*50)

In [ ]:
# Example 2.1b: Create prompt with custom question

prompt2 = Prompt(
    prompt_id="beach_001_q_what",
    image_id="beach_001",
    structure_word="what",
    question="Describe the main activity happening in this beach scene."
)

print("Prompt with custom question:")
print(f"  prompt_id: {prompt2.prompt_id}")
print(f"  question: {prompt2.question}")  # Custom question
print(f"  difficulty: {prompt2.difficulty}")  # Still auto-assigned

print("\n" + "="*50)

In [ ]:
# Example 2.1c: Create prompt with custom difficulty

prompt3 = Prompt(
    prompt_id="beach_001_q_mood",
    image_id="beach_001",
    structure_word="mood",
    question="What emotions does this scene evoke?",
    difficulty=3  # Override default
)

print("Prompt with custom question AND difficulty:")
print(f"  structure_word: {prompt3.structure_word}")
print(f"  question: {prompt3.question}")
print(f"  difficulty: {prompt3.difficulty}")  # Custom difficulty

print("\n✓ All Prompt variations created!")

## Function 2.2: `to_dict()`

### What it does:
Converts the Prompt object into a dictionary for saving to JSONL file.

### Parameters:
- None (uses object's data)

### Returns:
- Dictionary with all prompt fields

In [ ]:
# Example 2.2: Convert prompt to dictionary

prompt = Prompt(
    prompt_id="park_001_q_where",
    image_id="park_001",
    structure_word="where",
    question="Where is this scene taking place?"
)

# Convert to dictionary
prompt_dict = prompt.to_dict()

print("Prompt object converted to dictionary:")
print(json.dumps(prompt_dict, indent=2))

print("\n✓ Ready to save to JSONL file!")

## Prompt Summary

| Function/Feature | Purpose |
|------------------|----------|
| `DIFFICULTY_MAP` | Maps structure words to difficulty levels |
| `DEFAULT_TEMPLATES` | Pre-written questions for each structure word |
| `__init__()` | Create new prompt with auto-generation features |
| `to_dict()` | Convert prompt to dictionary for saving |

---

---
---
# CLASS 3: GoldAnswer

## What This Class Does:
The `GoldAnswer` class stores **the correct answer(s) for a prompt**. Key features:
- Links to a specific prompt via prompt_id
- Can store multiple acceptable answers
- Used as "ground truth" for evaluating AI responses

**Think of it as:** An answer key that accepts multiple correct variations.

---

## Function 3.1: `__init__()`

### What it does:
Creates a new GoldAnswer object with one or more correct answers.

### Parameters:
- `prompt_id` (str): ID of the prompt this answers
- `answers` (List[str]): List of acceptable answer strings

### Why list of answers?
Multiple phrasings can be correct: "a child", "a young girl", "a kid" are all valid answers to "Who is in the image?"

In [ ]:
# Example 3.1a: Create GoldAnswer with single answer

answer1 = GoldAnswer(
    prompt_id="beach_001_q_where",
    answers=["beach"]
)

print("GoldAnswer with single answer:")
print(f"  prompt_id: {answer1.prompt_id}")
print(f"  answers: {answer1.answers}")
print(f"  number of answers: {len(answer1.answers)}")

print("\n" + "="*50)

In [ ]:
# Example 3.1b: Create GoldAnswer with multiple acceptable answers

answer2 = GoldAnswer(
    prompt_id="beach_001_q_who",
    answers=[
        "family of four",
        "parents and two children",
        "a family",
        "four people",
        "mother, father, and two kids"
    ]
)

print("GoldAnswer with multiple acceptable answers:")
print(f"  prompt_id: {answer2.prompt_id}")
print(f"  number of answers: {len(answer2.answers)}")
print("\n  All acceptable answers:")
for i, ans in enumerate(answer2.answers, 1):
    print(f"    {i}. {ans}")

print("\n✓ GoldAnswer objects created!")

## Function 3.2: `to_dict()`

### What it does:
Converts the GoldAnswer object into a dictionary for saving to JSONL file.

### Parameters:
- None (uses object's data)

### Returns:
- Dictionary with prompt_id and answers list

In [ ]:
# Example 3.2: Convert GoldAnswer to dictionary

answer = GoldAnswer(
    prompt_id="park_001_q_what",
    answers=["playing", "children playing", "playground activities"]
)

# Convert to dictionary
answer_dict = answer.to_dict()

print("GoldAnswer object converted to dictionary:")
print(json.dumps(answer_dict, indent=2))

print("\n✓ Ready to save to JSONL file!")

## GoldAnswer Summary

| Function | Purpose | Input | Output |
|----------|---------|-------|--------|
| `__init__()` | Create answer object | prompt_id, answers list | GoldAnswer object |
| `to_dict()` | Convert to dictionary | None | Dict |

**Key advantage:** Supports multiple correct answer variations for flexible evaluation.

---

---
---
# CLASS 4: ImageCurator

## What This Class Does:
The `ImageCurator` class is the **master manager** that:
- Stores collections of images, prompts, and answers
- Loads data from CSV files
- Saves data to JSON/JSONL files
- Validates data completeness
- Exports data back to CSV
- Manages the entire dataset lifecycle

**Think of it as:** A complete database system for your image dataset.

### Internal Storage:
- `self.images` = Dictionary of {image_id: ImageMetadata}
- `self.prompts` = Dictionary of {prompt_id: Prompt}
- `self.gold_answers` = Dictionary of {prompt_id: GoldAnswer}

---

## Function 4.1: `__init__()`

### What it does:
Initializes the curator with file paths and empty storage containers.

### Parameters:
- `data_dir` (str): Directory where all data will be stored (default: "data")

### What gets set up:
- File paths for metadata.json, prompts.jsonl, gold_answers.jsonl
- Empty dictionaries for images, prompts, answers
- Creates data/images directory if it doesn't exist

In [ ]:
# Example 4.1: Initialize ImageCurator

curator = ImageCurator(data_dir="data")

print("ImageCurator initialized:")
print(f"  data_dir: {curator.data_dir}")
print(f"  images_dir: {curator.images_dir}")
print(f"  metadata_file: {curator.metadata_file}")
print(f"  prompts_file: {curator.prompts_file}")
print(f"  answers_file: {curator.answers_file}")
print(f"\n  images: {curator.images}")  # Empty dict
print(f"  prompts: {curator.prompts}")  # Empty dict
print(f"  gold_answers: {curator.gold_answers}")  # Empty dict

print("\n✓ Curator ready to manage dataset!")

## Function 4.2: `add_image()`

### What it does:
Adds a new image to the curator's collection.

### Parameters:
- `image_id` (str): Unique identifier for the image
- `file_path` (str): Path to the image file

### Effect:
Creates a new ImageMetadata object and stores it in `self.images` dictionary

In [ ]:
# Example 4.2: Add images to curator

curator = ImageCurator()

# Add first image
curator.add_image("beach_001", "data/images/beach.jpg")
print(f"Added beach_001. Total images: {len(curator.images)}")

# Add second image
curator.add_image("park_002", "data/images/park.jpg")
print(f"Added park_002. Total images: {len(curator.images)}")

# Add third image
curator.add_image("mountain_003", "data/images/mountain.jpg")
print(f"Added mountain_003. Total images: {len(curator.images)}")

print(f"\nAll images in curator:")
for img_id in curator.images.keys():
    print(f"  - {img_id}")

print("\n✓ Images added to curator!")

## Function 4.3: `set_scene_type()`

### What it does:
Sets the scene type for an image.

### Parameters:
- `image_id` (str): ID of the image to update
- `scene_type` (str): Scene type ("indoor", "outdoor", or "activity")

### Safety feature:
Only updates if image_id exists in the curator

In [ ]:
# Example 4.3: Set scene types

curator = ImageCurator()
curator.add_image("beach_001", "data/images/beach.jpg")
curator.add_image("kitchen_002", "data/images/kitchen.jpg")

# Set scene types
curator.set_scene_type("beach_001", "outdoor")
print(f"beach_001 scene_type: {curator.images['beach_001'].scene_type}")

curator.set_scene_type("kitchen_002", "indoor")
print(f"kitchen_002 scene_type: {curator.images['kitchen_002'].scene_type}")

# Try to set for non-existent image (safely does nothing)
curator.set_scene_type("nonexistent_999", "outdoor")
print("\n✓ Scene types set!")

## Function 4.4: `set_complexity()`

### What it does:
Sets the complexity level for an image.

### Parameters:
- `image_id` (str): ID of the image
- `complexity_level` (int): Complexity from 1-3 (1=simple, 3=complex)

In [ ]:
# Example 4.4: Set complexity levels

curator = ImageCurator()
curator.add_image("simple_001", "data/images/simple.jpg")
curator.add_image("medium_002", "data/images/medium.jpg")
curator.add_image("complex_003", "data/images/complex.jpg")

# Set different complexity levels
curator.set_complexity("simple_001", 1)
curator.set_complexity("medium_002", 2)
curator.set_complexity("complex_003", 3)

print("Complexity levels:")
for img_id, metadata in curator.images.items():
    print(f"  {img_id}: Level {metadata.complexity_level}")

print("\n✓ Complexity levels set!")

## Function 4.5: `set_subjects()`

### What it does:
Sets the primary subjects (what's in the image).

### Parameters:
- `image_id` (str): ID of the image
- `subjects` (List[str]): List of subjects

In [ ]:
# Example 4.5: Set subjects

curator = ImageCurator()
curator.add_image("beach_001", "data/images/beach.jpg")

# Set subjects as a list
curator.set_subjects("beach_001", ["family", "beach", "ocean", "sand"])

print(f"Subjects for beach_001:")
for subject in curator.images["beach_001"].primary_subjects:
    print(f"  - {subject}")

print("\n✓ Subjects set!")

## Function 4.6: `set_actions()`

### What it does:
Sets the actions happening in the image.

### Parameters:
- `image_id` (str): ID of the image
- `actions` (List[str]): List of actions

In [ ]:
# Example 4.6: Set actions

curator = ImageCurator()
curator.add_image("park_001", "data/images/park.jpg")

# Set actions
curator.set_actions("park_001", ["playing", "running", "swinging", "laughing"])

print(f"Actions in park_001:")
for action in curator.images["park_001"].actions:
    print(f"  - {action}")

print("\n✓ Actions set!")

## Function 4.7: `set_colors()`

### What it does:
Sets the prominent colors in the image.

### Parameters:
- `image_id` (str): ID of the image
- `colors` (List[str]): List of colors

In [ ]:
# Example 4.7: Set colors

curator = ImageCurator()
curator.add_image("sunset_001", "data/images/sunset.jpg")

# Set colors
curator.set_colors("sunset_001", ["orange", "pink", "purple", "gold"])

print(f"Colors in sunset_001:")
for color in curator.images["sunset_001"].colors:
    print(f"  - {color}")

print("\n✓ Colors set!")

## Function 4.8: `set_setting()`

### What it does:
Sets the setting/location description.

### Parameters:
- `image_id` (str): ID of the image
- `setting` (str): Setting description

In [ ]:
# Example 4.8: Set settings

curator = ImageCurator()
curator.add_image("beach_001", "data/images/beach.jpg")
curator.add_image("mountain_002", "data/images/mountain.jpg")

# Set settings
curator.set_setting("beach_001", "tropical beach at sunset")
curator.set_setting("mountain_002", "rocky mountain trail with pine trees")

print("Settings:")
for img_id, metadata in curator.images.items():
    print(f"  {img_id}: {metadata.setting}")

print("\n✓ Settings set!")

## Function 4.9: `set_structure_words()`

### What it does:
Sets structured answers for question types (who/what/where/color/size/mood).

### Parameters:
- `image_id` (str): ID of the image
- `structure_words` (Dict[str, str]): Dictionary of structure word answers

In [ ]:
# Example 4.9: Set structure words

curator = ImageCurator()
curator.add_image("beach_001", "data/images/beach.jpg")

# Set structure words
structure_words = {
    "who": "family of four",
    "what": "beach walk",
    "where": "sandy beach",
    "color": "blue, orange, golden",
    "size": "medium group",
    "mood": "peaceful and relaxed"
}

curator.set_structure_words("beach_001", structure_words)

print("Structure words for beach_001:")
for key, value in curator.images["beach_001"].structure_words.items():
    if value:  # Only show non-empty
        print(f"  {key:10} → {value}")

print("\n✓ Structure words set!")

## Function 4.10: `set_gold_standard()`

### What it does:
Sets the perfect, complete description of the image.

### Parameters:
- `image_id` (str): ID of the image
- `gold_standard` (str): Complete description

In [ ]:
# Example 4.10: Set gold standard descriptions

curator = ImageCurator()
curator.add_image("beach_001", "data/images/beach.jpg")

# Set the perfect description
curator.set_gold_standard(
    "beach_001",
    "A family of four walking along a sandy beach during a beautiful golden sunset, with waves gently lapping at the shore."
)

print("Gold standard for beach_001:")
print(f"  {curator.images['beach_001'].gold_standard}")

print("\n✓ Gold standard set!")

## Function 4.11: `add_prompt()`

### What it does:
Adds a Prompt object to the curator's collection.

### Parameters:
- `prompt` (Prompt): A Prompt object to add

### Effect:
Stores the prompt in `self.prompts` dictionary with prompt_id as key

In [ ]:
# Example 4.11: Add prompts to curator

curator = ImageCurator()
curator.add_image("beach_001", "data/images/beach.jpg")

# Create and add prompts
prompt1 = Prompt("beach_001_q_who", "beach_001", "who")
prompt2 = Prompt("beach_001_q_what", "beach_001", "what")
prompt3 = Prompt("beach_001_q_where", "beach_001", "where")

curator.add_prompt(prompt1)
curator.add_prompt(prompt2)
curator.add_prompt(prompt3)

print(f"Total prompts added: {len(curator.prompts)}")
print("\nPrompts in curator:")
for prompt_id, prompt in curator.prompts.items():
    print(f"  {prompt_id}: {prompt.question}")

print("\n✓ Prompts added!")

## Function 4.12: `add_gold_answer()`

### What it does:
Adds a GoldAnswer object to the curator's collection.

### Parameters:
- `answer` (GoldAnswer): A GoldAnswer object to add

### Effect:
Stores the answer in `self.gold_answers` dictionary with prompt_id as key

In [ ]:
# Example 4.12: Add gold answers to curator

curator = ImageCurator()

# Create and add gold answers
answer1 = GoldAnswer("beach_001_q_who", ["family", "family of four"])
answer2 = GoldAnswer("beach_001_q_what", ["walking", "beach walk"])
answer3 = GoldAnswer("beach_001_q_where", ["beach", "sandy beach"])

curator.add_gold_answer(answer1)
curator.add_gold_answer(answer2)
curator.add_gold_answer(answer3)

print(f"Total gold answers added: {len(curator.gold_answers)}")
print("\nGold answers in curator:")
for prompt_id, answer in curator.gold_answers.items():
    print(f"  {prompt_id}: {answer.answers}")

print("\n✓ Gold answers added!")

## Function 4.13: `save_metadata()`

### What it does:
Saves all image metadata to a JSON file.

### Parameters:
- None (uses curator's internal data)

### Effect:
Creates/overwrites `data/metadata.json` with all image metadata

### File format:
Single JSON file with all images as nested objects

In [ ]:
# Example 4.13: Save metadata to file

curator = ImageCurator()
curator.add_image("beach_001", "data/images/beach.jpg")
curator.set_scene_type("beach_001", "outdoor")
curator.set_complexity("beach_001", 2)
curator.set_subjects("beach_001", ["family", "beach"])
curator.set_gold_standard("beach_001", "A family at the beach.")

# Save to file
curator.save_metadata()
print("✓ Metadata saved to data/metadata.json")

# Read and display the file
with open('data/metadata.json', 'r') as f:
    saved_data = json.load(f)
    
print("\nSaved metadata:")
print(json.dumps(saved_data, indent=2))

## Function 4.14: `save_prompts()`

### What it does:
Saves all prompts to a JSONL file (one prompt per line).

### Parameters:
- None (uses curator's internal data)

### Effect:
Creates/overwrites `data/prompts.jsonl` with all prompts

### File format:
JSONL (JSON Lines) - each line is a separate JSON object

In [ ]:
# Example 4.14: Save prompts to file

curator = ImageCurator()
curator.add_image("beach_001", "data/images/beach.jpg")

# Add some prompts
prompt1 = Prompt("beach_001_q_who", "beach_001", "who")
prompt2 = Prompt("beach_001_q_what", "beach_001", "what")
curator.add_prompt(prompt1)
curator.add_prompt(prompt2)

# Save to file
curator.save_prompts()
print("✓ Prompts saved to data/prompts.jsonl")

# Read and display the file
print("\nSaved prompts (JSONL format):")
with open('data/prompts.jsonl', 'r') as f:
    for line in f:
        print(json.loads(line))

## Function 4.15: `save_gold_answers()`

### What it does:
Saves all gold answers to a JSONL file (one answer set per line).

### Parameters:
- None (uses curator's internal data)

### Effect:
Creates/overwrites `data/gold_answers.jsonl` with all answers

### File format:
JSONL - each line is a separate answer object

In [ ]:
# Example 4.15: Save gold answers to file

curator = ImageCurator()

# Add some answers
answer1 = GoldAnswer("beach_001_q_who", ["family", "family of four"])
answer2 = GoldAnswer("beach_001_q_what", ["walking", "beach walk"])
curator.add_gold_answer(answer1)
curator.add_gold_answer(answer2)

# Save to file
curator.save_gold_answers()
print("✓ Gold answers saved to data/gold_answers.jsonl")

# Read and display the file
print("\nSaved gold answers (JSONL format):")
with open('data/gold_answers.jsonl', 'r') as f:
    for line in f:
        print(json.loads(line))

## Function 4.16: `load_from_csv()`

### What it does:
**The most powerful function!** Loads all data from a CSV file and automatically:
- Creates ImageMetadata for each row
- Generates Prompts for each structure word
- Creates GoldAnswers from provided answers or structure word values

### Parameters:
- `csv_file` (str): Name of CSV file in data directory (default: "image_input.csv")

### Returns:
- Tuple of (loaded_count, error_count)

### CSV Format:
Uses semicolons (;) to separate multiple values in a cell

In [ ]:
# Example 4.16: Load from CSV file

# First, create a sample CSV file
sample_csv_data = """image_id,filename,scene_type,complexity,subjects,actions,colors,setting,structure_who,structure_what,structure_where,gold_standard,answer_who,answer_what
beach_001,beach.jpg,outdoor,2,family;beach,walking;playing,blue;orange,sandy beach,family of four,beach walk,beach,A family walking on a sandy beach.,family;parents and children,walking;beach walk
park_002,park.jpg,outdoor,2,children;playground,playing;swinging,green;red,city park,children,playing,park,Children playing in a park.,children;kids,playing;having fun"""

# Write sample CSV
with open('data/sample_images.csv', 'w') as f:
    f.write(sample_csv_data)

print("Sample CSV created!\n")

# Now load it
curator = ImageCurator()
loaded, errors = curator.load_from_csv("sample_images.csv")

print(f"✓ Loaded {loaded} images")
print(f"  Errors: {errors}")
print(f"\nDataset contents:")
print(f"  Images: {len(curator.images)}")
print(f"  Prompts: {len(curator.prompts)}")
print(f"  Gold Answers: {len(curator.gold_answers)}")

# Show what was loaded
print("\nImages loaded:")
for img_id in curator.images.keys():
    print(f"  - {img_id}")

print("\n✓ CSV loaded successfully!")

## Function 4.17: `export_to_csv()`

### What it does:
Exports all current data back to a CSV file for review/editing.

### Parameters:
- `csv_file` (str): Output filename (default: "metadata_export.csv")

### Effect:
Creates a CSV file with all images, metadata, prompts, and answers

In [ ]:
# Example 4.17: Export to CSV

curator = ImageCurator()
curator.add_image("beach_001", "data/images/beach.jpg")
curator.set_scene_type("beach_001", "outdoor")
curator.set_complexity("beach_001", 2)
curator.set_subjects("beach_001", ["family", "beach"])
curator.set_gold_standard("beach_001", "Family at beach.")

# Add a prompt and answer
prompt = Prompt("beach_001_q_who", "beach_001", "who")
answer = GoldAnswer("beach_001_q_who", ["family"])
curator.add_prompt(prompt)
curator.add_gold_answer(answer)

# Export to CSV
curator.export_to_csv("my_export.csv")
print("✓ Exported to data/my_export.csv")

# Show first few lines
print("\nExported CSV preview:")
with open('data/my_export.csv', 'r') as f:
    for i, line in enumerate(f):
        if i < 3:  # Show first 3 lines
            print(line.strip()[:100] + "...")

## Function 4.18: `validate_dataset()`

### What it does:
Checks if all images have complete required metadata.

### Parameters:
- None (checks curator's internal data)

### Returns:
- List of error messages (empty list if all valid)

### Checks for:
- scene_type present
- complexity_level set
- subjects listed
- actions listed
- colors listed
- setting described
- gold_standard provided

In [ ]:
# Example 4.18a: Validate incomplete dataset

curator = ImageCurator()
curator.add_image("incomplete_001", "data/images/incomplete.jpg")
curator.set_scene_type("incomplete_001", "outdoor")
# Missing: complexity, subjects, actions, colors, setting, gold_standard

errors = curator.validate_dataset()

if errors:
    print(f"❌ Validation found {len(errors)} errors:")
    for error in errors:
        print(f"  - {error}")
else:
    print("✓ All images valid!")

print("\n" + "="*50)

In [ ]:
# Example 4.18b: Validate complete dataset

curator = ImageCurator()
curator.add_image("complete_001", "data/images/complete.jpg")
curator.set_scene_type("complete_001", "outdoor")
curator.set_complexity("complete_001", 2)
curator.set_subjects("complete_001", ["family"])
curator.set_actions("complete_001", ["walking"])
curator.set_colors("complete_001", ["blue"])
curator.set_setting("complete_001", "beach")
curator.set_gold_standard("complete_001", "Complete description.")

errors = curator.validate_dataset()

if errors:
    print(f"❌ Validation found {len(errors)} errors")
else:
    print("✓ All images valid! No errors found.")
    print("\nAll required fields present:")
    print("  ✓ scene_type")
    print("  ✓ complexity_level")
    print("  ✓ subjects")
    print("  ✓ actions")
    print("  ✓ colors")
    print("  ✓ setting")
    print("  ✓ gold_standard")

## Function 4.19: `print_status()`

### What it does:
Prints a nicely formatted status report of all images.

### Parameters:
- None (uses curator's internal data)

### Shows:
- Total counts (images, prompts, answers)
- Completion status for each image (✓ or ✗)
- Key metadata for each image
- Number of prompts per image

In [ ]:
# Example 4.19: Print status report

curator = ImageCurator()

# Add complete image
curator.add_image("beach_001", "data/images/beach.jpg")
curator.set_scene_type("beach_001", "outdoor")
curator.set_complexity("beach_001", 2)
curator.set_subjects("beach_001", ["family", "beach"])
curator.set_actions("beach_001", ["walking"])
curator.set_colors("beach_001", ["blue", "orange"])
curator.set_setting("beach_001", "beach")
curator.set_gold_standard("beach_001", "Family at beach.")

# Add prompts
prompt1 = Prompt("beach_001_q_who", "beach_001", "who")
prompt2 = Prompt("beach_001_q_what", "beach_001", "what")
curator.add_prompt(prompt1)
curator.add_prompt(prompt2)

# Add incomplete image
curator.add_image("incomplete_002", "data/images/incomplete.jpg")
curator.set_scene_type("incomplete_002", "indoor")

# Print status
curator.print_status()

## ImageCurator Function Summary

| # | Function | Purpose |
|---|----------|----------|
| 1 | `__init__()` | Initialize curator |
| 2 | `add_image()` | Add image to collection |
| 3 | `set_scene_type()` | Set scene type |
| 4 | `set_complexity()` | Set complexity level |
| 5 | `set_subjects()` | Set subjects list |
| 6 | `set_actions()` | Set actions list |
| 7 | `set_colors()` | Set colors list |
| 8 | `set_setting()` | Set setting description |
| 9 | `set_structure_words()` | Set structured answers |
| 10 | `set_gold_standard()` | Set gold description |
| 11 | `add_prompt()` | Add prompt to collection |
| 12 | `add_gold_answer()` | Add answer to collection |
| 13 | `save_metadata()` | Save metadata to JSON |
| 14 | `save_prompts()` | Save prompts to JSONL |
| 15 | `save_gold_answers()` | Save answers to JSONL |
| 16 | `load_from_csv()` | Load all data from CSV |
| 17 | `export_to_csv()` | Export all data to CSV |
| 18 | `validate_dataset()` | Check data completeness |
| 19 | `print_status()` | Print status report |

---

---
---
# COMPLETE WORKFLOW EXAMPLE

## Using All Classes and Functions Together

In [ ]:
# Complete workflow demonstration

print("="*70)
print("COMPLETE WORKFLOW: All Classes and Functions")
print("="*70)

# STEP 1: Create curator
print("\n[1] Creating ImageCurator...")
curator = ImageCurator(data_dir="data")
print("    ✓ Curator initialized")

# STEP 2: Add images manually
print("\n[2] Adding images...")
curator.add_image("beach_001", "data/images/beach.jpg")
curator.add_image("park_002", "data/images/park.jpg")
print(f"    ✓ Added {len(curator.images)} images")

# STEP 3: Set metadata for first image
print("\n[3] Setting metadata for beach_001...")
curator.set_scene_type("beach_001", "outdoor")
curator.set_complexity("beach_001", 2)
curator.set_subjects("beach_001", ["family", "beach", "ocean"])
curator.set_actions("beach_001", ["walking", "playing"])
curator.set_colors("beach_001", ["blue", "orange", "gold"])
curator.set_setting("beach_001", "sandy beach at sunset")
curator.set_structure_words("beach_001", {
    "who": "family of four",
    "what": "beach walk",
    "where": "beach"
})
curator.set_gold_standard("beach_001", "A family walking on a beach at sunset.")
print("    ✓ Metadata set")

# STEP 4: Create and add prompts
print("\n[4] Creating prompts...")
prompt1 = Prompt("beach_001_q_who", "beach_001", "who")
prompt2 = Prompt("beach_001_q_what", "beach_001", "what")
prompt3 = Prompt("beach_001_q_where", "beach_001", "where")
curator.add_prompt(prompt1)
curator.add_prompt(prompt2)
curator.add_prompt(prompt3)
print(f"    ✓ Added {len(curator.prompts)} prompts")

# STEP 5: Create and add gold answers
print("\n[5] Creating gold answers...")
answer1 = GoldAnswer("beach_001_q_who", ["family", "family of four"])
answer2 = GoldAnswer("beach_001_q_what", ["walking", "beach walk"])
answer3 = GoldAnswer("beach_001_q_where", ["beach", "sandy beach"])
curator.add_gold_answer(answer1)
curator.add_gold_answer(answer2)
curator.add_gold_answer(answer3)
print(f"    ✓ Added {len(curator.gold_answers)} gold answers")

# STEP 6: Validate dataset
print("\n[6] Validating dataset...")
errors = curator.validate_dataset()
if errors:
    print(f"    ⚠ Found {len(errors)} errors")
    for error in errors[:3]:
        print(f"      - {error}")
else:
    print("    ✓ All images valid")

# STEP 7: Save everything
print("\n[7] Saving to files...")
curator.save_metadata()
print("    ✓ Saved metadata.json")
curator.save_prompts()
print("    ✓ Saved prompts.jsonl")
curator.save_gold_answers()
print("    ✓ Saved gold_answers.jsonl")

# STEP 8: Export to CSV
print("\n[8] Exporting to CSV...")
curator.export_to_csv("workflow_export.csv")
print("    ✓ Exported to workflow_export.csv")

# STEP 9: Print status
print("\n[9] Status Report:")
curator.print_status()

print("\n" + "="*70)
print("✓ WORKFLOW COMPLETE!")
print("="*70)
print(f"\nFinal counts:")
print(f"  Images: {len(curator.images)}")
print(f"  Prompts: {len(curator.prompts)}")
print(f"  Gold Answers: {len(curator.gold_answers)}")

---
# COMPLETE FUNCTION REFERENCE

## All Classes and Their Functions

### Class 1: ImageMetadata (3 functions)
1. `__init__(image_id, file_path)` - Create new metadata object
2. `to_dict()` - Convert to dictionary
3. `from_dict(data)` - Create from dictionary

### Class 2: Prompt (1 function + 2 class variables)
- `DIFFICULTY_MAP` - Difficulty levels mapping
- `DEFAULT_TEMPLATES` - Default question templates
1. `__init__(prompt_id, image_id, structure_word, question, difficulty)` - Create prompt
2. `to_dict()` - Convert to dictionary

### Class 3: GoldAnswer (2 functions)
1. `__init__(prompt_id, answers)` - Create answer object
2. `to_dict()` - Convert to dictionary

### Class 4: ImageCurator (19 functions)
1. `__init__(data_dir)` - Initialize curator
2. `add_image(image_id, file_path)` - Add image
3. `set_scene_type(image_id, scene_type)` - Set scene type
4. `set_complexity(image_id, complexity_level)` - Set complexity
5. `set_subjects(image_id, subjects)` - Set subjects
6. `set_actions(image_id, actions)` - Set actions
7. `set_colors(image_id, colors)` - Set colors
8. `set_setting(image_id, setting)` - Set setting
9. `set_structure_words(image_id, structure_words)` - Set structure words
10. `set_gold_standard(image_id, gold_standard)` - Set gold standard
11. `add_prompt(prompt)` - Add prompt
12. `add_gold_answer(answer)` - Add answer
13. `save_metadata()` - Save metadata to JSON
14. `save_prompts()` - Save prompts to JSONL
15. `save_gold_answers()` - Save answers to JSONL
16. `load_from_csv(csv_file)` - Load from CSV
17. `export_to_csv(csv_file)` - Export to CSV
18. `validate_dataset()` - Validate completeness
19. `print_status()` - Print status report

**Total: 4 classes, 27 functions/methods**

---

## Congratulations!
You now understand every single class and function in image_curator.py! 🎉